In [2]:
import pandas as pd
df = pd.read_csv(r'F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\combine_results.csv')
df


,draft_year,nfl_id,combine_position,combine_height,combine_weight,hand_size,arm_length,wing_span,ten_yd_split,forty,vertical,broad_jump,three_cone,short_shuttle,bench_reps,ngs_athleticism_score,ngs_college_production_score,ngs_final_score
0,2023,56040,TE,77.875,245,10.000,32.250,79.750,1.60,4.84,38.5,125.0,NaN,NaN,NaN,60,NaN,NaN
1,2023,56010,DB,70.625,191,8.750,29.625,72.250,1.58,4.59,33.5,116.0,NaN,NaN,23.0,60,77.0,74.0
2,2023,56085,TE,79.375,255,10.250,34.000,83.125,1.57,4.55,40.0,128.0,6.87,4.12,23.0,97,65.0,79.0
3,2023,56056,WR,72.250,192,9.625,31.875,76.125,1.51,4.33,NaN,NaN,NaN,NaN,NaN,76,68.0,73.0
4,2023,56118,WR,71.625,191,9.500,31.000,75.000,1.52,4.54,38.5,120.0,6.98,4.15,14.0,65,65.0,64.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505,2025,58259,OL,78.000,335,10.250,33.875,83.125,1.82,5.26,29.5,107.0,7.77,NaN,NaN,72,75.0,75.0
506,2025,58454,OL,77.750,309,10.000,32.000,77.125,1.78,5.11,28.5,110.0,7.78,4.63,NaN,80,58.0,66.0
507,2025,58261,OL,80.125,316,10.375,33.000,81.000,1.77,5.21,NaN,106.0,7.71,4.70,22.0,73,67.0,73.0
508,2025,58896,OL,79.125,309,10.500,33.000,82.875,1.82,5.24,26.0,103.0,NaN,NaN,NaN,61,54.0,56.0


In [3]:
df.columns.tolist()

['draft_year',
 'nfl_id',
 'combine_position',
 'combine_height',
 'combine_weight',
 'hand_size',
 'arm_length',
 'wing_span',
 'ten_yd_split',
 'forty',
 'vertical',
 'broad_jump',
 'three_cone',
 'short_shuttle',
 'bench_reps',
 'ngs_athleticism_score',
 'ngs_college_production_score',
 'ngs_final_score']

### Initial Data Profiling — combine_results.csv

**Objective:** Examine the structure and overall quality of NFL Combine results.

**Checks:**
- Dataset dimensions and data types
- Missing values and completeness percentages
- Duplicate records and player identifiers
- Distribution of Combine positions and draft years

**Purpose:** Identify potential data quality issues before conducting WR-specific analysis and cross-table validation.

In [4]:
# 1. Dataset dimensions
print("Dataset Shape:", df.shape)

# 2. Data types and memory usage
df.info(memory_usage="deep")

# 3. Missing values
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (
        df.isna().mean() * 100
    ).round(2)
})

display(missing_summary)

# 4. Duplicate checks
print("Duplicate Rows:", df.duplicated().sum())
print("Duplicate NFL IDs:", df["nfl_id"].duplicated().sum())

# 5. Draft year distribution
print("\nDraft Year Distribution:")
print(df["draft_year"].value_counts(dropna=False).sort_index())

# 6. Position distribution
print("\nCombine Position Distribution:")
print(df["combine_position"].value_counts(dropna=False))

Dataset Shape: (510, 18)
<class 'pandas.DataFrame'>
RangeIndex: 510 entries, 0 to 509
Data columns (total 18 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   draft_year                    510 non-null    int64  
 1   nfl_id                        510 non-null    int64  
 2   combine_position              510 non-null    str    
 3   combine_height                510 non-null    float64
 4   combine_weight                510 non-null    int64  
 5   hand_size                     510 non-null    float64
 6   arm_length                    510 non-null    float64
 7   wing_span                     510 non-null    float64
 8   ten_yd_split                  425 non-null    float64
 9   forty                         424 non-null    float64
 10  vertical                      438 non-null    float64
 11  broad_jump                    423 non-null    float64
 12  three_cone                    180 non-null    floa

,missing_count,missing_percentage
draft_year,0,0.00
nfl_id,0,0.00
combine_position,0,0.00
combine_height,0,0.00
combine_weight,0,0.00
hand_size,0,0.00
arm_length,0,0.00
wing_span,0,0.00
ten_yd_split,85,16.67
forty,86,16.86


Duplicate Rows: 0
Duplicate NFL IDs: 0

Draft Year Distribution:
draft_year
2023    160
2024    177
2025    173
Name: count, dtype: int64

Combine Position Distribution:
combine_position
DB    122
OL    121
DL    118
WR    107
TE     42
Name: count, dtype: int64


### Cross-Table Validation — Players vs Combine Results

**Objective:** Validate player identity and position consistency between `players.csv` and `combine_results.csv`.

**Checks:**
- Compare player IDs across both datasets.
- Identify unmatched players.
- Compare WR classifications between `nfl_position` and `combine_position`.

**Purpose:** Detect potential inconsistencies that may affect WR filtering and subsequent dataset integration.

**Note:** Different position classifications do not automatically indicate incorrect data.

In [5]:

# Adjust paths based on your project structure
players_df = pd.read_csv(r"F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\players.csv")
combine_df = df

# Compare player IDs
players_ids = set(players_df["nfl_id"])
combine_ids = set(combine_df["nfl_id"])

print("Players IDs:", len(players_ids))
print("Combine IDs:", len(combine_ids))

print(
    "IDs in players but not combine:",
    len(players_ids - combine_ids)
)

print(
    "IDs in combine but not players:",
    len(combine_ids - players_ids)
)

# Compare WR classifications
position_comparison = players_df[
    ["nfl_id", "display_name", "nfl_position"]
].merge(
    combine_df[["nfl_id", "combine_position"]],
    on="nfl_id",
    how="outer",
    indicator=True
)

wr_mismatch = position_comparison[
    (
        (position_comparison["nfl_position"] == "WR")
        !=
        (position_comparison["combine_position"] == "WR")
    )
]

print("\nWR Classification Differences:")
display(wr_mismatch)

Players IDs: 510
Combine IDs: 510
IDs in players but not combine: 0
IDs in combine but not players: 0

WR Classification Differences:


,nfl_id,display_name,nfl_position,combine_position,_merge
93,56062,Elijah Higgins,TE,WR,both


### Cross-Table Validation Findings

**Player ID Integrity**
- Total unique player IDs in `players.csv`: 510
- Total unique player IDs in `combine_results.csv`: 510
- Unmatched player IDs: 0
- Player ID coverage: 100%

**Position Consistency**
- WR players in `players.csv`: 106
- WR players in `combine_results.csv`: 107
- One WR classification difference was identified:
  - Elijah Higgins (`nfl_id = 56062`)
  - Players position: TE
  - Combine position: WR

**Assessment:** The position difference may reflect a positional transition or different classification conventions rather than incorrect data.

**Decision:** Use `players.nfl_position` as the reference for WR cohort selection, while preserving the original Combine position.

**Status:** Documented; no data modification required.

### WR-Specific Combine Data Completeness

**Objective:** Evaluate the availability of Combine measurements for the 106 Wide Receivers identified in `players.csv`.

**Method:** Filter Combine results using the WR player IDs from the players dataset and calculate missing counts and percentages.

**Purpose:** Identify which Combine measurements are sufficiently complete for potential use in WR-focused analysis and modeling.

**Note:** Missing measurements will be documented without imputation during the profiling stage.

In [6]:
# Identify WR players from players.csv
wr_ids = players_df.loc[
    players_df["nfl_position"] == "WR",
    "nfl_id"
]

# Filter combine data using the reference WR IDs
wr_combine = combine_df[
    combine_df["nfl_id"].isin(wr_ids)
].copy()

# Calculate missing values for WRs
wr_missing_summary = pd.DataFrame({
    "missing_count": wr_combine.isna().sum(),
    "missing_percentage": (
        wr_combine.isna().mean() * 100
    ).round(2)
})

print("Total WR Combine Records:", len(wr_combine))

display(
    wr_missing_summary[
        wr_missing_summary["missing_count"] > 0
    ].sort_values(
        "missing_percentage",
        ascending=False
    )
)

Total WR Combine Records: 106


,missing_count,missing_percentage
bench_reps,78,73.58
three_cone,74,69.81
short_shuttle,68,64.15
broad_jump,12,11.32
vertical,11,10.38
forty,10,9.43
ten_yd_split,10,9.43


### Final Data Profiling Summary — combine_results.csv

**Dataset Overview**
- Total records: 510
- Total columns: 18
- Unique player IDs: 510
- Duplicate records: 0
- Draft years: 2023–2025

**Data Completeness**
- High missingness was identified in `three_cone` (64.71%), `bench_reps` (62.35%), and `short_shuttle` (60.59%).
- Moderate missingness was observed in several other athletic measurements.
- One missing value was found in each of `ngs_college_production_score` and `ngs_final_score`.

**Cross-Table Validation**
- All 510 player IDs match between `players.csv` and `combine_results.csv`.
- One WR classification difference was identified for Elijah Higgins (`nfl_id = 56062`), who is classified as TE in `players.csv` and WR in `combine_results.csv`.
- `players.nfl_position` will be used to define the WR population.

**WR-Specific Findings**
- Total WR records: 106
- High missingness:
  - `bench_reps`: 73.58%
  - `three_cone`: 69.81%
  - `short_shuttle`: 64.15%
- Lower missingness:
  - `broad_jump`: 11.32%
  - `vertical`: 10.38%
  - `forty`: 9.43%
  - `ten_yd_split`: 9.43%

**Recommendations**
- Preserve missing values during profiling.
- Investigate missingness patterns before choosing an imputation or feature-selection strategy.
- Preserve original Combine position classifications.
- Validate measurement ranges and units before using Combine features in modeling.

**Status:** Initial profiling completed. Measurement validity checks remain pending.



In [7]:
measurements = [
    "combine_height",
    "combine_weight",
    "hand_size",
    "arm_length",
    "wing_span",
    "ten_yd_split",
    "forty",
    "vertical",
    "broad_jump",
    "three_cone",
    "short_shuttle",
    "bench_reps"
]

display(combine_df[measurements].describe().T)

,count,mean,std,min,25%,50%,75%,max
combine_height,510.0,74.579657,2.628102,68.125,72.6250,75.000,76.50000,80.625
combine_weight,510.0,248.380392,53.254702,154.000,198.0000,245.500,304.00000,374.000
hand_size,510.0,9.571814,0.602178,8.000,9.1250,9.500,10.00000,11.625
arm_length,510.0,32.410784,1.419286,28.500,31.3750,32.375,33.50000,36.875
wing_span,510.0,78.465931,3.273243,69.000,76.2500,78.500,80.84375,88.250
ten_yd_split,425.0,1.632000,0.115573,1.460,1.5400,1.590,1.74000,1.960
forty,424.0,4.733514,0.309449,4.280,4.4700,4.620,5.01000,5.480
vertical,438.0,33.810502,4.141179,23.000,31.0000,34.000,37.00000,44.000
broad_jump,423.0,118.151300,9.064842,93.000,111.0000,120.000,125.00000,138.000
three_cone,180.0,7.310611,0.390453,6.570,6.9975,7.290,7.62250,8.190


### Measurement Validity Assessment

**Method:** Descriptive statistics were used to examine the minimum, maximum, and distribution of Combine measurements.

**Findings:**
- All 12 reviewed measurement columns have plausible minimum and maximum values.
- No non-positive measurement values were observed.
- No obvious range anomalies were identified.
- Extreme values may reflect differences in player positions rather than data quality errors.

**Limitations:**
- Measurement units have not been independently verified against the official data dictionary.
- Statistical range checks do not guarantee record-level accuracy.
- Position-specific outlier analysis may be performed during preprocessing.

**Conclusion:** Initial measurement range validation is complete. No immediate corrections are required based on the reviewed statistics.

**Overall Status:** Initial data profiling completed; documented missingness and remaining validation limitations will be addressed during preprocessing and modeling.